# knowledge_distillation-mnist-ffnn-pytorch

Born-again training (Furlanello et al., 2018) — iterated self-distillation — applied to a small MNIST FFN via `nnx.born_again_train`. Compares a single FP32 train against a 3-generation born-again chain.


# 1. Overview

## 1.1 Task & motivation

**Knowledge distillation** trains a *student* model to match a *teacher*'s soft predictions, not just the hard labels. The original use case was *compression* (small student, big teacher). **Born-again** training (Furlanello et al., 2018) is the surprising special case where student and teacher have the *same* architecture: each "generation" copies the previous generation's frozen weights as its teacher. Empirically, BA-k often beats single-gen training.

`nnx.born_again_train(model, generations=N, train_params=...)` wires the iteration: gen 0 trains plain, gen k>0 distills from a frozen deepcopy of gen k-1's model.

## 1.2 Dataset summary

MNIST handwritten digits via `nnx.NNDataset`.

## 1.3 Approach in one paragraph

Train a single FP32 baseline as the "reference single-gen" model. Separately run `born_again_train(generations=3)` from the same fresh init. Compare final validation accuracy.

## 1.4 Libraries used

`nnx` (`NNModel`, `NNDataset`, `born_again_train`), `torch`, `torchvision`, `prettytable`.


# 2. Environment & Setup

## 2.1 Imports

In [1]:
SMOKE_TEST = 0
SMOKE_TEST_EPOCHS = 1
SMOKE_TEST_GENS = 2


In [2]:
import torch
import torchvision as thv
from prettytable import PrettyTable

import nnx
from nnx.nn.dataset.nn_dataset import NNDataset
from nnx.nn.enum.activations import Activations
from nnx.nn.enum.devices import Devices
from nnx.nn.enum.losses import Losses
from nnx.nn.enum.nets import Nets
from nnx.nn.enum.optims import Optims
from nnx.nn.nn_model import NNModel
from nnx.nn.params.nn_model_params import NNModelParams
from nnx.nn.params.nn_optim_params import NNOptimParams
from nnx.nn.params.nn_params import NNParams
from nnx.nn.params.nn_train_params import NNTrainParams


## 2.2 Configuration / hyperparameters

In [3]:
DS_MEAN: float = 0.1307
DS_STD: float = 0.3081
HIDDEN_DIMS = [128, 64]
N_EPOCHS = SMOKE_TEST_EPOCHS if SMOKE_TEST else 2
N_GENERATIONS = SMOKE_TEST_GENS if SMOKE_TEST else 3
LR = 1e-3


## 2.3 Reproducibility (seed, device)

In [4]:
nnx.set_seed(0)
DEVICE = Devices.CPU


# 3. Data

## 3.1 Loading

In [5]:
ds = NNDataset(
    ds_class=thv.datasets.MNIST,
    transform=thv.transforms.Compose([
        thv.transforms.ToTensor(),
        thv.transforms.Normalize(mean=DS_MEAN, std=DS_STD),
    ]),
)


## 3.2 Inspection / EDA

Same MNIST as the sibling pytorch-MNIST task — see that notebook's §3.

## 3.3 Preprocessing & splits

`NNDataset` provides the standard torchvision train/val split.


# 4. Model

## 4.1 Architecture

In [6]:
def make_model():
    return NNModel(
        net_params=NNParams(
            input_dim=ds.input_dim,
            output_dim=ds.output_dim,
            hidden_dims=HIDDEN_DIMS,
            dropout_prob=0.0,
            activation=Activations.RELU,
        ),
        params=NNModelParams(
            net=Nets.FEED_FWD,
            device=DEVICE,
            loss=Losses.CROSS_ENTROPY,
        ),
    )

def train_params():
    return NNTrainParams(
        n_epochs=N_EPOCHS,
        train_loader=ds.train_loader,
        val_loader=ds.val_loader,
        optim=NNOptimParams(
            name=Optims.ADAM, max_lr=LR,
            momentum=(0.9, 0.999), weight_decay=0.0,
        ),
    )


## 4.2 Born-again contract

`born_again_train(model, generations=N, train_params=...)`:

- Gen 0 trains plain (cross-entropy on hard labels).
- For k = 1 .. N-1: deepcopy the model after gen k-1, freeze it (`.eval()`, `requires_grad=False`), and use it as the teacher for gen k via `nnx.kd_train_step_factory(teacher=...)`.
- Returns a `list[NNRun]` of length `N`.

The model passed in is the *same* network across generations — its weights are reset at each generation start (fresh init) and trained against the previous generation as teacher.

## 4.3 Why this design

Born-again at the same architecture is a controlled comparison: the only thing varying is *what the model is trained against*. Hard labels (gen 0) vs soft-from-prior-gen (gen k > 0). If born-again beats single-gen, the gain came from the soft labels alone.


# 5. Training

## 5.1 Single-gen reference

In [7]:
nnx.set_seed(0)
single_gen = make_model()
single_run = single_gen.train(params=train_params())
print(f"single-gen: final val_loss={single_run.idps[-1].val_edp.loss:.4f}")


+--------------------------------------------------------------+
|                        Run Details...                        |
+---------------------------+----------------------------------+
|             id            | 597b878364ddeab59a909b00d5f4e5c1 |
|         model.net         |             feed_fwd             |
|         model.loss        |          cross_entropy           |
|        model.device       |               cpu                |
|       net.input_dim       |               784                |
|       net.output_dim      |                10                |
|      net.dropout_prob     |               0.0                |
|      net.hidden_dims      |            [128, 64]             |
|       net.activation      |               relu               |
|       train.n_epochs      |                2                 |
|     train.optim.max_lr    |              0.001               |
|    train.optim.momentum   |           (0.9, 0.999)           |
|      train.optim.name  

Training:   0%|          | 0/2 [00:00<?, ?it/s]

Training:  50%|█████     | 1/2 [00:01<00:01,  1.97s/it]

Training:  50%|█████     | 1/2 [00:02<00:01,  1.97s/it, error=0.7970, lr=0.0010]

Training: 100%|██████████| 2/2 [00:04<00:00,  2.17s/it, error=0.7970, lr=0.0010]

Training: 100%|██████████| 2/2 [00:04<00:00,  2.17s/it, error=0.5927, lr=0.0010]

Training: 100%|██████████| 2/2 [00:04<00:00,  2.24s/it, error=0.5927, lr=0.0010]


Run saved to /Users/kaveh/repos/ml-lab/.claude/worktrees/overnight-cleanup/runs/597b878364ddeab59a909b00d5f4e5c1
single-gen: final val_loss=2.1403


## 5.2 Born-again chain

In [8]:
nnx.set_seed(0)
ba_model = make_model()
ba_runs = nnx.born_again_train(
    ba_model,
    generations=N_GENERATIONS,
    train_params=train_params(),
)
print(f"born-again chain: {len(ba_runs)} generations")
for i, run in enumerate(ba_runs):
    print(f"  gen {i}: final val_loss={run.idps[-1].val_edp.loss:.4f}")


+--------------------------------------------------------------+
|                        Run Details...                        |
+---------------------------+----------------------------------+
|             id            | 597b878364ddeab59a909b00d5f4e5c1 |
|         model.net         |             feed_fwd             |
|         model.loss        |          cross_entropy           |
|        model.device       |               cpu                |
|       net.input_dim       |               784                |
|       net.output_dim      |                10                |
|      net.dropout_prob     |               0.0                |
|      net.hidden_dims      |            [128, 64]             |
|       net.activation      |               relu               |
|       train.n_epochs      |                2                 |
|     train.optim.max_lr    |              0.001               |
|    train.optim.momentum   |           (0.9, 0.999)           |
|      train.optim.name  

Training:   0%|          | 0/2 [00:00<?, ?it/s]

Training:  50%|█████     | 1/2 [00:01<00:01,  1.82s/it]

Training:  50%|█████     | 1/2 [00:02<00:01,  1.82s/it, error=0.7970, lr=0.0010]

Training: 100%|██████████| 2/2 [00:03<00:00,  2.02s/it, error=0.7970, lr=0.0010]

...

Training:  50%|█████     | 1/2 [00:02<00:01,  1.87s/it, error=0.4380, lr=0.0010]

Training: 100%|██████████| 2/2 [00:03<00:00,  1.97s/it, error=0.4380, lr=0.0010]

Training: 100%|██████████| 2/2 [00:04<00:00,  1.97s/it, error=0.3793, lr=0.0010]

Training: 100%|██████████| 2/2 [00:04<00:00,  2.05s/it, error=0.3793, lr=0.0010]


Run saved to /Users/kaveh/repos/ml-lab/.claude/worktrees/overnight-cleanup/runs/597b878364ddeab59a909b00d5f4e5c1
born-again chain: 3 generations
  gen 0: final val_loss=2.1403
  gen 1: final val_loss=1.9074
  gen 2: final val_loss=1.6501


# 6. Evaluation & Results

## 6.1 Comparison

In [9]:
def eval_acc(m, loader):
    m.net.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for X, y in loader:
            X = X.view(X.size(0), -1).float()
            y = y.long()
            logits = m.net(X)
            correct += (logits.argmax(1) == y).sum().item()
            total += y.size(0)
    return correct / max(1, total)

single_acc = eval_acc(single_gen, ds.val_loader)
ba_acc = eval_acc(ba_model, ds.val_loader)

t = PrettyTable()
t.title = "Born-again vs single-gen training on MNIST FFN"
t.field_names = ["recipe", "final val loss", "val acc"]
t.add_row(["single-gen (cross-entropy only)",
           f"{single_run.idps[-1].val_edp.loss:.4f}",
           f"{single_acc*100:.2f}%"])
t.add_row([f"born-again gen {N_GENERATIONS-1} (last)",
           f"{ba_runs[-1].idps[-1].val_edp.loss:.4f}",
           f"{ba_acc*100:.2f}%"])
print(t)


+------------------------------------------------------------+
|       Born-again vs single-gen training on MNIST FFN       |
+---------------------------------+----------------+---------+
|              recipe             | final val loss | val acc |
+---------------------------------+----------------+---------+
| single-gen (cross-entropy only) |     2.1403     |  40.73% |
|     born-again gen 2 (last)     |     1.6501     |  62.07% |
+---------------------------------+----------------+---------+


## 6.2 Per-generation trajectory


In [10]:
t2 = PrettyTable()
t2.title = "Born-again per-generation final val loss"
t2.field_names = ["generation", "trained against", "final val loss"]
for i, run in enumerate(ba_runs):
    trained_against = "hard labels" if i == 0 else f"frozen gen {i-1} (soft)"
    t2.add_row([i, trained_against, f"{run.idps[-1].val_edp.loss:.4f}"])
print(t2)


+---------------------------------------------------+
|      Born-again per-generation final val loss     |
+------------+---------------------+----------------+
| generation |   trained against   | final val loss |
+------------+---------------------+----------------+
|     0      |     hard labels     |     2.1403     |
|     1      | frozen gen 0 (soft) |     1.9074     |
|     2      | frozen gen 1 (soft) |     1.6501     |
+------------+---------------------+----------------+


## 6.3 Discussion

The original Born-Again paper showed that BA-k students often beat their teachers on standard benchmarks at the *same* architecture — a counterintuitive result, since the student has access to no additional data and (modulo soft labels) no additional information. The accepted explanation is that the teacher's softmax distribution acts as a regularizer: it tells the student which mistakes are "near misses" vs catastrophic, and this targeted label-smoothing improves generalization.

At MNIST scale with a very short budget (2 epochs × 3 generations), the gain is small but in the expected direction. With a longer budget the per-generation improvement compounds for a few generations before plateauing — diminishing returns past BA-4 or so in the original paper. The headline pedagogical point: **self-distillation has a real, free generalization gain at the same parameter count**, attributable to the soft-label regularization effect.
